In [1]:
from openai import OpenAI

# Modify OpenAI's API key and API base to use vLLM's API server.
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)
completion = client.completions.create(model="Qwen/Qwen2.5-1.5B-Instruct",
                                    prompt="San Francisco is a")
print("Completion result:", completion)

Completion result: Completion(id='cmpl-19eaf5d251404b9a80843123aeab49ad', choices=[CompletionChoice(finish_reason='length', index=0, logprobs=None, text=' city in the state of California. It is located on the San Francisco Peninsula,', stop_reason=None, prompt_logprobs=None)], created=1756937221, model='Qwen/Qwen2.5-1.5B-Instruct', object='text_completion', system_fingerprint=None, usage=CompletionUsage(completion_tokens=16, prompt_tokens=4, total_tokens=20, completion_tokens_details=None, prompt_tokens_details=None), service_tier=None, kv_transfer_params=None)


In [2]:
completion.choices[0].text

' city in the state of California. It is located on the San Francisco Peninsula,'

In [36]:
import asyncio, httpx, os, json
import nest_asyncio

# Apply the nest_asyncio patch to allow nested event loops
nest_asyncio.apply()

BASE = "http://localhost:8001/v1" #os.environ.get("BASE_URL", "http://SERVER_IP:8000/v1")
HEADERS = {"Authorization": f"Bearer {os.environ.get('VLLM_API_KEY','sk-local-123')}",
           "Content-Type": "application/json"}

async def one(i):
    payload = {
        "model":"Qwen/Qwen2.5-1.5B-Instruct",
        "messages":[{"role":"user","content":f"Say hi from task {i} in 1 short sentence."}],
        "temperature":0.2,
        "stream": False
    }
    async with httpx.AsyncClient(timeout=60) as client:
        r = await client.post(f"{BASE}/chat/completions", headers=HEADERS, json=payload)
        r.raise_for_status()
        return i, r.json()["choices"][0]["message"]["content"]

async def main():
    results = await asyncio.gather(*(one(i) for i in range(1000)))
    for i, text in results[:5]:
        print(i, text)

# Run the main coroutine using asyncio.run() within a Jupyter notebook
await main()


0 Hello! I'm here to assist with your tasks.
1 Hello! I'm here to help with your tasks.
2 Hello! I'm here to help with your tasks.
3 Hello! I'm here to help with your tasks.
4 Hello! I'm here to help with your tasks.


In [35]:
import asyncio, httpx, os, json

BASE = "http://localhost:8000/v1"
HEADERS = {"Authorization": f"Bearer {os.environ.get('VLLM_API_KEY','sk-local-123')}",
           "Content-Type": "application/json"}

async def main():
    limits = httpx.Limits(max_connections=2000, max_keepalive_connections=2000)
    async with httpx.AsyncClient(timeout=60, limits=limits) as client:
        sem = asyncio.Semaphore(512)  # cap in-flight requests

        async def one(i):
            payload = {
                "model":"Qwen/Qwen2.5-1.5B-Instruct",
                "messages":[{"role":"user","content":f"Say hi from task *{i}* in 1 short sentence."}],
                "temperature":0.2,
                "stream": False,
                # (Optional) make it more GPU-bound to see scaling:
                # "max_tokens": 64
            }
            async with sem:
                r = await client.post(f"{BASE}/chat/completions", headers=HEADERS, json=payload)
                r.raise_for_status()
                return i, r.json()["choices"][0]["message"]["content"]

        results = await asyncio.gather(*(one(i) for i in range(2000)))
        print(results[:5])

asyncio.run(main())


ReadError: 

In [38]:
from openai import OpenAI
client = OpenAI(base_url="http://34.12.108.233:8000/v1", api_key="")
r = client.chat.completions.create(model="Qwen/Qwen2.5-7B-Instruct",
                                   messages=[{"role": "user", "content": "Hello!"}])
print(r.choices[0].message.content)


Hello! How can I assist you today?
